In [ ]:
import geopandas as gpd
import pandas as pd
import planetary_computer
from pystac_client import Client
from datetime import datetime, timedelta

# --- USER PARAMETERS ---
TILE_SHAPEFILE = r"D:\WRI\Field Boundaries\WRI Mexico\sentinel_2_index_shapefile_WRImexicotestsite.shp"
OUTPUT_CSV = r"D:\WRI\Field Boundaries\WRI Mexico\sentinel2_tileindex_WRImexicotestsite2024.csv"
STAC_API_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
# Connect to STAC API
stac_client = Client.open(STAC_API_URL, modifier=planetary_computer.sign_inplace)

# Read shapefile containing Sentinel-2 tile IDs
gdf = gpd.read_file(TILE_SHAPEFILE)
if gdf.empty:
    raise ValueError("Error: The shapefile contains no valid data.")

# Extract unique Sentinel-2 tile IDs from the "Name" column
tile_ids = gdf["Name"].unique().tolist()
print(f"Number of manually selected Sentinel-2 tiles: {len(tile_ids)}")

# --- FUNCTION TO FIND IMAGE PAIRS ---
def find_sentinel_pairs(tile_id, min_months=5, max_cloud=0):
    """
    Finds two Sentinel-2 image pairs for a given tile.
    If no pairs are found, relaxes constraints step by step.
    """
    start_date = datetime(2020, 1, 1)
    end_date = datetime(2020, 12, 31)

    # Search conditions with progressively relaxed cloud cover and time gap
    conditions = [
        (max_cloud, min_months),  # 0% cloud, 5+ months
        (1, min_months),          # 1% cloud, 5+ months
        (0, 3),                   # 0% cloud, 3+ months
        (1, 3),                   # 1% cloud, 3+ months
        (2, 3),                    # 2% cloud, 3+ months
        (3, 3),
        (4, 3),                 # below more were added for tropical regions
        (5, 3)#,
        #(6, 5),
        #(7, 5),
        #(8, 5),
        #(9, 4),
        #(10, 4),
        #(11, 4),
        #(12, 4),
        #(10, 3),
        #(11, 3),
        #(12, 3)
    ]

    for cloud, months in conditions:
        print(f"Searching for {tile_id}: Cloud ≤ {cloud}%, Min Gap = {months} months")

        # Query STAC API for Sentinel-2 imagery for this tile
        search = stac_client.search(
            collections=["sentinel-2-l2a"],
            datetime=f"{start_date.strftime('%Y-%m-%d')}/{end_date.strftime('%Y-%m-%d')}",
            query={
                "s2:mgrs_tile": {"eq": tile_id},  # Ensure images are for the correct tile
                "eo:cloud_cover": {"lte": cloud}  # Filter by cloud cover
            },
            limit=100
        )

        items = list(search.items())

        # Sort images by date
        items.sort(key=lambda x: x.datetime)

        # Find pairs with required time gap
        pairs = []
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                date1 = items[i].datetime
                date2 = items[j].datetime

                if abs((date2 - date1).days) >= months * 30:
                    pairs.append((items[i].id, items[j].id))
                    if len(pairs) >= 2:
                        return pairs  # Return once two pairs are found

    print(f"No valid pairs found for {tile_id}")
    return pairs

# --- PROCESS EACH TILE ---
results = []

for tile_id in tile_ids:
    pairs = find_sentinel_pairs(tile_id)
    for pair in pairs:
        results.append([tile_id, pair[0], pair[1]])

# Convert to DataFrame and save as CSV
df_results = pd.DataFrame(results, columns=["Tile", "Window A", "Window B"])
df_results.to_csv(OUTPUT_CSV, index=False)

print(f"Results saved to {OUTPUT_CSV}")


Number of manually selected Sentinel-2 tiles: 45
Searching for 20HLG: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HLH: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HMD: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HME: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HMF: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HMG: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HMH: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HNC: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HND: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HNE: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HNF: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HNG: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HNH: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HNJ: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HNK: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HPC: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HPD: Cloud ≤ 0%, Min Gap = 5 months
Searching for 20HPE: Cloud ≤ 0%, Min Gap = 5 months
Searching for 2

Add a no-data filter percentage

In [ ]:
import geopandas as gpd
import pandas as pd
import planetary_computer
from pystac_client import Client
from datetime import datetime

# --- USER PARAMETERS ---
TILE_SHAPEFILE = r"D:\WRI\Field Boundaries\WRI Mexico\sentinel_2_index_shapefile_WRImexicotestsite.shp"
OUTPUT_CSV = r"D:\WRI\Field Boundaries\WRI Mexico\sentinel2_tileindex_WRImexicotestsite2024.csv"
STAC_API_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

# Connect to STAC API
stac_client = Client.open(STAC_API_URL, modifier=planetary_computer.sign_inplace)

# Read shapefile containing Sentinel-2 tile IDs
gdf = gpd.read_file(TILE_SHAPEFILE)
if gdf.empty:
    raise ValueError("Error: The shapefile contains no valid data.")

tile_ids = gdf["Name"].unique().tolist()
print(f"Number of manually selected Sentinel-2 tiles: {len(tile_ids)}")

# --- FUNCTION TO FIND IMAGE PAIRS ---
def find_sentinel_pairs(tile_id, min_months=5, max_cloud=0):
    """
    Finds two Sentinel-2 image pairs for a given tile.
    Returns a list of tuples: (img1_id, img2_id, img1_nodata, img2_nodata)
    """
    start_date = datetime(2024, 1, 1)
    end_date = datetime(2024, 12, 31)

    conditions = [
        (max_cloud, min_months),
        (1, min_months),
        (0, 3),
        (1, 3),
        (2, 3),
        (3, 3),
        (4, 3),
        (5, 3)
    ]

    for cloud, months in conditions:
        print(f"Searching for {tile_id}: Cloud ≤ {cloud}%, Min Gap = {months} months")

        search = stac_client.search(
            collections=["sentinel-2-l2a"],
            datetime=f"{start_date.strftime('%Y-%m-%d')}/{end_date.strftime('%Y-%m-%d')}",
            query={
                "s2:mgrs_tile": {"eq": tile_id},
                "eo:cloud_cover": {"lte": cloud},
                "s2:nodata_pixel_percentage": {"lt": 100}
            },
            limit=100
        )

        items = list(search.items())
        items.sort(key=lambda x: x.datetime)

        pairs = []
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                date1 = items[i].datetime
                date2 = items[j].datetime

                if abs((date2 - date1).days) >= months * 30:
                    id1 = items[i].id
                    id2 = items[j].id
                    nodata1 = items[i].properties.get("s2:nodata_pixel_percentage", None)
                    nodata2 = items[j].properties.get("s2:nodata_pixel_percentage", None)
                    pairs.append((id1, id2, nodata1, nodata2))
                    if len(pairs) >= 2:
                        return pairs
    print(f"No valid pairs found for {tile_id}")
    return pairs

# --- PROCESS EACH TILE ---
results = []

for tile_id in tile_ids:
    pairs = find_sentinel_pairs(tile_id)
    for pair in pairs:
        results.append([tile_id, pair[0], pair[1], pair[2], pair[3]])

# Save results to CSV
df_results = pd.DataFrame(results, columns=["Tile", "Window A", "Window B", "Nodata % A", "Nodata % B"])
df_results.to_csv(OUTPUT_CSV, index=False)

print(f"Results saved to {OUTPUT_CSV}")


Number of manually selected Sentinel-2 tiles: 6
Searching for 14PQC: Cloud ≤ 0%, Min Gap = 5 months
Searching for 14QQD: Cloud ≤ 0%, Min Gap = 5 months
Searching for 14QQE: Cloud ≤ 0%, Min Gap = 5 months
Searching for 14QRD: Cloud ≤ 0%, Min Gap = 5 months
Searching for 14QRE: Cloud ≤ 0%, Min Gap = 5 months
Searching for 15QTU: Cloud ≤ 0%, Min Gap = 5 months
Results saved to D:\WRI\Field Boundaries\WRI Mexico\sentinel2_tileindex_WRImexicotestsite.csv


In [1]:
import geopandas as gpd
import os

# Define input GeoJSON and output Shapefile path
input_geojson = r"C:\Users\grupp\Downloads\polygon_from_image.geojson"
output_dir = r"C:\Users\grupp\Downloads"
output_shapefile = os.path.join(output_dir, "polygon_from_image.shp")

# Read GeoJSON
gdf = gpd.read_file(input_geojson)

# Export to Shapefile
gdf.to_file(output_shapefile, driver="ESRI Shapefile")

print(f"Shapefile saved to: {output_shapefile}")


Shapefile saved to: C:\Users\grupp\Downloads\polygon_from_image.shp


Manual List Version:

In [3]:
import pandas as pd
import planetary_computer
from pystac_client import Client
from datetime import datetime, timedelta

# --- USER PARAMETERS ---
TILE_IDS = ["37MBU", "37MBV", "37MCU"]  # Manually specified Sentinel-2 tile IDs
OUTPUT_CSV = r"D:\WRI_Kenya_sentinel2_tiles_and_pairs_2020.csv"
STAC_API_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

# Connect to STAC API
stac_client = Client.open(STAC_API_URL, modifier=planetary_computer.sign_inplace)

print(f"Number of manually selected Sentinel-2 tiles: {len(TILE_IDS)}")

# --- FUNCTION TO FIND IMAGE PAIRS ---
def find_sentinel_pairs(tile_id, min_months=5, max_cloud=0):
    """
    Finds two Sentinel-2 image pairs for a given tile.
    If no pairs are found, relaxes constraints step by step.
    """
    start_date = datetime(2020, 1, 1)
    end_date = datetime(2020, 12, 31)

    # Search conditions with progressively relaxed cloud cover and time gap
    conditions = [
        (max_cloud, min_months),  # 0% cloud, 5+ months
        (1, min_months),          # 1% cloud, 5+ months
        (0, 3),                   # 0% cloud, 3+ months
        (1, 3),                   # 1% cloud, 3+ months
        (2, 3)                    # 2% cloud, 3+ months
    ]

    for cloud, months in conditions:
        print(f"Searching for {tile_id}: Cloud ≤ {cloud}%, Min Gap = {months} months")

        # Query STAC API for Sentinel-2 imagery for this tile
        search = stac_client.search(
            collections=["sentinel-2-l2a"],
            datetime=f"{start_date.strftime('%Y-%m-%d')}/{end_date.strftime('%Y-%m-%d')}",
            query={
                "s2:mgrs_tile": {"eq": tile_id},  # Ensure images are for the correct tile
                "eo:cloud_cover": {"lte": cloud}  # Filter by cloud cover
            },
            limit=100
        )

        items = list(search.items())

        # Sort images by date
        items.sort(key=lambda x: x.datetime)

        # Find pairs with required time gap
        pairs = []
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                date1 = items[i].datetime
                date2 = items[j].datetime

                if abs((date2 - date1).days) >= months * 30:
                    pairs.append((items[i].id, items[j].id))
                    if len(pairs) >= 2:
                        return pairs  # Return once two pairs are found

    print(f"No valid pairs found for {tile_id}")
    return pairs

# --- PROCESS EACH TILE ---
results = []

for tile_id in TILE_IDS:
    pairs = find_sentinel_pairs(tile_id)
    for pair in pairs:
        results.append([tile_id, pair[0], pair[1]])

# Convert to DataFrame and save as CSV
df_results = pd.DataFrame(results, columns=["Tile", "Window A", "Window B"])
df_results.to_csv(OUTPUT_CSV, index=False)

print(f"Results saved to {OUTPUT_CSV}")


Number of manually selected Sentinel-2 tiles: 3
Searching for 37MBU: Cloud ≤ 0%, Min Gap = 5 months
Searching for 37MBU: Cloud ≤ 1%, Min Gap = 5 months
Searching for 37MBU: Cloud ≤ 0%, Min Gap = 3 months
Searching for 37MBU: Cloud ≤ 1%, Min Gap = 3 months
Searching for 37MBU: Cloud ≤ 2%, Min Gap = 3 months
No valid pairs found for 37MBU
Searching for 37MBV: Cloud ≤ 0%, Min Gap = 5 months
Searching for 37MCU: Cloud ≤ 0%, Min Gap = 5 months
Searching for 37MCU: Cloud ≤ 1%, Min Gap = 5 months
Searching for 37MCU: Cloud ≤ 0%, Min Gap = 3 months
Searching for 37MCU: Cloud ≤ 1%, Min Gap = 3 months
Searching for 37MCU: Cloud ≤ 2%, Min Gap = 3 months
No valid pairs found for 37MCU
Results saved to D:\WRI_Kenya_sentinel2_tiles_and_pairs_2020.csv
